First, let's get our libraries going and import the files that we need to utilize for this assignment. Additionally, we will preform some of the base level clean ups that we were asked. First preform our database merger, then drop the columns that we do not need. 


In [1]:
import pandas as pd
from datetime import datetime

# Load datasets
hospitals = pd.read_csv("data/CaliforniaHospitalData.csv")
personnel = pd.read_csv("data/CaliforniaHospitalData_Personnel.txt", delimiter="\t")

# The needed Merge on 'HospitalID'
merged = pd.merge(hospitals, personnel, on="HospitalID")

# Drop specified columns as instructed
merged.drop(columns=["Work_ID", "PositionID", "Website"], inplace=True)


Next, we shall create the filter desired for the hospitals with 15 plus beds, and a positive OI. Then export the data to a csv file. 

In [ ]:
# Filter Small/Rural with 15+ beds and positive operating income
filtered = merged[
    (merged["Teaching"] == "Small/Rural") &
    (merged["AvlBeds"] >= 15) &
    (merged["OperInc"] >= 0)
]

# Export 
filtered.to_csv("data/hospital_data_new.txt", sep="\t", index=False)

Let's read in our new file. Then preform the column renames. 

In [ ]:
# Reload the filtered data to rename the columns
renamed = pd.read_csv("data/hospital_data_new.txt", delimiter="\t")

# Rename columns
renamed.rename(columns={
    "NoFTE": "FullTimeCount",
    "NetPatRev": "NetPatientRevenue",
    "InOperExp": "InpatientOperExp",
    "OutOperExp": "OutpatientOperExp",
    "OperRev": "Operating_Revenue",
    "OperInc": "Operating_Income"
}, inplace=True)

Now we can create two new rows that will be used to insert myself as an employee. Followed with a simple match and concatenate to create this entry. 

In [4]:
# Create new rows
today = datetime.today().strftime("%Y-%m-%d")

# Insert myself as a employee in two seperate hospitals
new_rows = pd.DataFrame([
    {
        "HospitalID": 17718,
        "LastName": "Kittle",
        "FirstName": "Brandon",
        "Gender": "M",
        "PositionTitle": "Acting Director",
        "Compensation": 248904,
        "MaxTerm": 8,
        "StartDate": today,
        "Phone": "406-555-1234",
        "Email": "brandon.kittle@edu.org"
    },
    {
        "HospitalID": 33207,
        "LastName": "Kittle",
        "FirstName": "Brandon",
        "Gender": "M",
        "PositionTitle": "Regional Representative",
        "Compensation": 46978,
        "MaxTerm": 4,
        "StartDate": today,
        "Phone": "406-555-5678",
        "Email": "bkittle@edu.org"
    }
])

# Match structure and concatenate
new_merge = pd.concat([renamed, new_rows], ignore_index=True)

Filtered Data: Non-Profit, > 250 FTE, unless NetPatientRevenue < 109000

In [5]:
filtered_np = new_merge[
    (new_merge["TypeControl"] == "Non Profit") &
    ((new_merge["FullTimeCount"] > 250) | (new_merge["NetPatientRevenue"] < 109000))
].drop(columns=["FirstName", "LastName", "Gender", "PositionTitle", "Compensation", "MaxTerm", "StartDate", "Phone", "Email"])


Filtered Data: Regional Representatives w/ Operating Income > 100,000

In [6]:
regional_reps = new_merge[
    (new_merge["PositionTitle"] == "Regional Representative") &
    (new_merge["Operating_Income"] > 100000)
]


Lastly, we shall preform the needed steps to convert and "confirm" such work done. 

In [8]:
# StartDate to datetime
new_merge["StartDate"] = pd.to_datetime(new_merge["StartDate"], format='mixed')

# Confirm 
print(new_merge.dtypes)
print(new_merge["StartDate"].head())

HospitalID                    int64
Name                         object
Zip                          object
TypeControl                  object
Teaching                     object
DonorType                    object
FullTimeCount               float64
NetPatientRevenue           float64
InpatientOperExp            float64
OutpatientOperExp           float64
Operating_Revenue           float64
Operating_Income            float64
AvlBeds                     float64
LastName                     object
FirstName                    object
Gender                       object
PositionTitle                object
Compensation                  int64
MaxTerm                       int64
StartDate            datetime64[ns]
Phone                        object
Email                        object
dtype: object
0   2009-01-01
1   2011-01-01
2   2011-01-01
3   2012-01-01
4   2009-01-01
Name: StartDate, dtype: datetime64[ns]
